In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

project_path = "/content/drive/MyDrive/Study/HKU/POLI3148 Data Science in Politics and Public Administration/Assignments/Assignment 1/"

if os.path.exists(project_path):
    print("Path found:", project_path)
    print("\nContents:")
    for item in os.listdir(project_path):
        print("-", item)
else:
    print("Path not found:", project_path)

# 1. Domestic Violence and Fatalities in Iran (2016-2026)

## Process Data

In [ ]:
import pandas as pd
import os
import glob

data_raw_path = os.path.join(project_path, "Data", "Data_Raw")
data_processed_path = os.path.join(project_path, "Data", "Data_Processed")

files = [
    "number_of_demonstration_events_by_country-year_as-of-03Ap.xlsx",
    "number_of_events_targeting_civilians_by_country-year_as-o.xlsx",
    "number_of_political_violence_events_by_country-year_as-of-03Apr2026.xlsx",
    "number_of_reported_civilian_fatalities_by_country-year_as.xlsx",
    "number_of_reported_fatalities_by_country-year_as-of-03Apr.xlsx"
]

all_iran_data = []

for file in files:
    file_path = os.path.join(data_raw_path, file)
    if os.path.exists(file_path):
        df = pd.read_excel(file_path)
        
        # Find the exact names of the required columns in this specific file
        country_col = [c for c in df.columns if c.lower() == 'country']
        year_col = [c for c in df.columns if c.lower() == 'year']
        val_col = [c for c in df.columns if c.upper() in ['EVENTS', 'FATALITIES']]
        
        if country_col and year_col and val_col: # Make sure we found all necessary parts
            # Filter for Iran
            df_iran = df[df[country_col[0]] == 'Iran'].copy()
            
            # Create a clean DataFrame strictly with the 4 columns requested
            clean_df = pd.DataFrame({
                'Country': df_iran[country_col[0]],
                'Year': df_iran[year_col[0]],
                'NUMBER': df_iran[val_col[0]],
                'Source_File': file
            })
            
            all_iran_data.append(clean_df)
        else:
            print(f"Missing required columns in: {file}")
            print(f"Found columns: {df.columns.tolist()}")
    else:
        print(f"File not found: {file_path}")

if all_iran_data:
    final_df = pd.concat(all_iran_data, ignore_index=True)
    out_path = os.path.join(data_processed_path, "Iran_violence.csv")
    final_df.to_csv(out_path, index=False)
    print(f"Data successfully filtered and consolidated to {out_path}")
    print(final_df.head(3)) # Show a quick preview to confirm the 4 columns
else:
    print("No data extracted.")

## Visualization

First visualize the trend in number of demonstration, political_violence', strategic_developments, events targeting_civilians, reported_fatalities, civilian_fatalities

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import os

if os.path.exists(out_path):
    ir_df = pd.read_csv(out_path)
    
    # Check what columns exist to plot Yearly aggregation
    year_col = [c for c in ir_df.columns if 'year' in c.lower()]
    val_cols = [c for c in ir_df.columns if c.lower() not in ['country', 'iso', 'region', 'source_file', 'year']]
    
    if year_col and val_cols:
        y_col = year_col[0]
        numeric_cols = ir_df.select_dtypes(include='number').columns
        val_sum = [c for c in numeric_cols if c != y_col]
        
        yearly_data = ir_df.groupby([y_col, 'Source_File'])[val_sum].sum().reset_index()
        
        # --- Add Strategic Developments from Iran_disobedience.csv ---
        csv_file_path = os.path.join(project_path, "Data", "Data_Processed", "Iran_disobedience.csv")
        if os.path.exists(csv_file_path):
            disob_df = pd.read_csv(csv_file_path)
            if 'WEEK' in disob_df.columns and 'EVENT_TYPE' in disob_df.columns:
                disob_df['year_extracted'] = pd.to_datetime(disob_df['WEEK']).dt.year
                strat_df = disob_df[disob_df['EVENT_TYPE'].str.contains('Strategic developments', na=False, case=False)]
                strat_yearly = strat_df.groupby('year_extracted')['EVENTS'].sum().reset_index()
                strat_yearly['Source_File'] = 'strategic_developments'
                strat_yearly.rename(columns={'year_extracted': y_col, 'EVENTS': val_sum[0]}, inplace=True)
                yearly_data = pd.concat([yearly_data, strat_yearly], ignore_index=True)
        # -------------------------------------------------------------

        def get_clean_label(filename):
            name = filename.lower()
            if 'demonstration' in name: return 'Demonstration Events'
            elif 'civilians' in name and 'fatalities' not in name: return 'Events Targeting Civilians'
            elif 'political_violence' in name: return 'Political Violence Events'
            elif 'strategic_developments' in name: return 'Strategic Developments'
            elif 'civilian_fatalities' in name: return 'Reported Civilian Fatalities'
            elif 'reported_fatalities' in name: return 'Total Fatalities'
            return filename[:20]
        
        # Explicitly order the sources: demonstration, violence, strategic, targeting civilians, then fatalities
        ordered_keywords = [
            'demonstration',
            'political_violence',
            'strategic_developments',
            'targeting_civilians',
            'reported_fatalities',
            'civilian_fatalities'
        ]
        
        unique_sources = []
        for kw in ordered_keywords:
            for sf in yearly_data['Source_File'].unique():
                if kw in sf.lower() and sf not in unique_sources:
                    unique_sources.append(sf)
        # Catch any remaining ones just in case
        for sf in yearly_data['Source_File'].unique():
            if sf not in unique_sources:
                unique_sources.append(sf)
                
        # Calculate maximum Y value for the "events" datasets to share the same scale
        event_sfs = [sf for sf in unique_sources if any(k in sf.lower() for k in ['demonstration', 'political_violence', 'strategic', 'targeting_civilians'])]
        max_events_y = yearly_data[yearly_data['Source_File'].isin(event_sfs)][val_sum[0]].max() if event_sfs else 0
        
        # Calculate shared Y value bounds for the two "fatalities" datasets
        fatalities_sfs = [sf for sf in unique_sources if sf not in event_sfs]
        if fatalities_sfs:
            fat_subset = yearly_data[yearly_data['Source_File'].isin(fatalities_sfs)][val_sum[0]]
            min_fat_y = fat_subset.min()
            max_fat_y = fat_subset.max()
            fat_y_margin = (max_fat_y - min_fat_y) * 0.15 if max_fat_y > min_fat_y else max_fat_y * 0.15
            shared_fat_bottom = max(0, min_fat_y - fat_y_margin)
            shared_fat_top = max_fat_y + fat_y_margin
        else:
            shared_fat_bottom, shared_fat_top = 0, 100
        
        # ACLED-like muted regional colors extended to 6
        colors = ['#62778d', '#8a4053', '#8e44ad', '#719e9c', '#cc5c4b', '#488c5e']
        
        def hex_to_rgba(hex_color, alpha=0.15):
            hex_color = hex_color.lstrip('#')
            return f"rgba({int(hex_color[0:2], 16)}, {int(hex_color[2:4], 16)}, {int(hex_color[4:6], 16)}, {alpha})"

        # Determine grid size (e.g., 2 rows x 3 columns for 6 items)
        n_cols = 3
        n_rows = int(np.ceil(len(unique_sources) / n_cols))
        
        subplot_titles = [get_clean_label(sf) for sf in unique_sources]
        
        # Reduce horizontal spacing since inner Y-axes will be hidden
        fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=subplot_titles, vertical_spacing=0.2, horizontal_spacing=0.04)
        
        for idx, sf in enumerate(unique_sources):
            row = (idx // n_cols) + 1
            col = (idx % n_cols) + 1
            
            subset = yearly_data[yearly_data['Source_File'] == sf].sort_values(y_col)
            
            if not subset.empty and len(val_sum) > 0:
                color = colors[idx % len(colors)]
                clean_label = get_clean_label(sf)
                
                y_max = max_events_y * 1.15 if sf in event_sfs else shared_fat_top
                y_min = 0 if sf in event_sfs else shared_fat_bottom
                
                # Plot the line with markers enabled for hover but styled specifically
                fig.add_trace(go.Scatter(
                    x=subset[y_col],
                    y=subset[val_sum[0]],
                    mode='lines',
                    line=dict(color=color, width=2.5),
                    marker=dict(size=8, color="white", line=dict(color=color, width=2.5)), # Hollow white circle on hover
                    fill='tozeroy' if y_min == 0 else 'tonexty',
                    fillcolor=hex_to_rgba(color, 0.25),
                    name=clean_label,
                    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}<extra></extra>',
                    hoverlabel=dict(bgcolor="white", font_size=13, bordercolor=color)
                ), row=row, col=col)
                
                fig.update_yaxes(range=[y_min, y_max], row=row, col=col)
                
                # Dynamic Y-axes logic: Only show labels and title on the first column
                if col == 1:
                    row_label = 'Number of Events' if row == 1 else 'Number of Deaths'
                    fig.update_yaxes(title_text=row_label, title_font=dict(size=12, color='#555555'), showticklabels=True, row=row, col=col)
                else:
                    fig.update_yaxes(showticklabels=False, row=row, col=col)
                
                fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray', zeroline=False, tickformat=",d", tickfont=dict(color='#555555'), row=row, col=col)
                fig.update_xaxes(showline=True, linewidth=1.5, linecolor='lightgray', showgrid=False, tickmode='linear', dtick=2, tickfont=dict(color='#555555'), row=row, col=col)

                # Add a vertical dashed line for the 2024 turning point
                fig.add_shape(
                    type="line",
                    x0=2024, x1=2024,
                    y0=y_min, y1=y_max,
                    line=dict(color="#333333", width=1.2, dash="dash"),
                    opacity=0.7,
                    row=row, col=col
                )
                
                

        # Fix subplot titles: modify strictly the generated annotations
        for i, annot in enumerate(fig.layout.annotations):
            if i < len(subplot_titles):
                annot.font.color = colors[i % len(colors)]
                annot.font.size = 14
                annot.text = f"<b>{subplot_titles[i]}</b>"
                
        fig.update_layout(
            title=dict(
                text="<b>Trends of Domestic Violence and Fatalities in Iran (2016-2026)</b>",
                font=dict(size=20, color='black', family="sans-serif"),
                x=0.03,
                y=0.95
            ),
            showlegend=False,
            plot_bgcolor="white",
            paper_bgcolor="white",
            height=350 * n_rows,
            margin=dict(t=100, b=80, l=80, r=40),
            hovermode="closest" # Change from 'x unified' to closest to remove the vertical line
        )
        
        # Safely append footnote annotation instead of overwriting all annotations
        fig.add_annotation(
            text="Data covers reported ACLED events up to 2026 March.",
            showarrow=False,
            xref="paper", yref="paper",
            x=0.01, y=-0.15,
            font=dict(size=11, color="gray", style="italic")
        )
        
        # Explicit baseline trace to avoid fill bleeding when 'tonexty' is used without lower boundary
        for idx, sf in enumerate(unique_sources):
            if sf not in event_sfs:
                row = (idx // n_cols) + 1
                col = (idx % n_cols) + 1
                subset = yearly_data[yearly_data['Source_File'] == sf].sort_values(y_col)
                if not subset.empty:
                    fig.add_trace(go.Scatter(x=subset[y_col], y=[shared_fat_bottom]*len(subset), mode='lines', line=dict(color='rgba(0,0,0,0)', width=0), hoverinfo='skip', showlegend=False), row=row, col=col)
                    # Re-order the traces so the baseline is before the filled trace
                    fig.data = list(fig.data[:-2]) + [fig.data[-1], fig.data[-2]]
                
        trend_chart_html = fig.to_html(include_plotlyjs=False, full_html=False)
        fig.show()
    else:
        print("Required 'Year' or 'Value' columns missing for visualization.")
else:
    print(f"Data file not found at {out_path} for visualization.")

Then compare trends in peaceful demonstration, protest with intervention, excessive force against protesters and violent demonstration - both number and fatalities

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import os

out_disobedience_path = os.path.join(project_path, "Data", "Data_Processed", "Iran_disobedience.csv")

if os.path.exists(out_disobedience_path):
    df_disob = pd.read_csv(out_disobedience_path)

    # Categories
    categories = [
        'Peaceful protest',
        'Protest with intervention',
        'Excessive force against protesters',
        'Violent demonstration'
    ]

    # Filter
    df_filtered = df_disob[df_disob['SUB_EVENT_TYPE'].isin(categories)].copy()

    # Extract Year from the WEEK column
    if 'WEEK' in df_filtered.columns:
        df_filtered['Year'] = pd.to_datetime(df_filtered['WEEK']).dt.year
        
        # Group by year and sub event type (now summing both EVENTS and FATALITIES)
        yearly_trend = df_filtered.groupby(['Year', 'SUB_EVENT_TYPE'])[['EVENTS', 'FATALITIES']].sum().reset_index()

        fig_trend = make_subplots(specs=[[{"secondary_y": True}]])
        
        # Specific colors mapping your previous definition
        colors_map = {
            'Peaceful protest': '#3498db',
            'Protest with intervention': '#f39c12',
            'Excessive force against protesters': '#8e44ad',
            'Violent demonstration': '#e74c3c'
        }

        for cat in categories:
            cat_data = yearly_trend[yearly_trend['SUB_EVENT_TYPE'] == cat].sort_values("Year")
            if not cat_data.empty:
                # Add Events trace
                fig_trend.add_trace(go.Scatter(
                    x=cat_data['Year'],
                    y=cat_data['EVENTS'],
                    mode='lines+markers',
                    name=cat,
                    line=dict(color=colors_map.get(cat, '#333333'), width=3),
                    marker=dict(size=8, color="white", line=dict(color=colors_map.get(cat, '#333333'), width=2)),
                    hovertemplate='Events: %{y:,.0f}<extra></extra>',
                    legendgroup=cat
                ), secondary_y=False)
                
                # Add Fatalities trace (Stacked Bar)
                fig_trend.add_trace(go.Bar(
                    x=cat_data['Year'],
                    y=cat_data['FATALITIES'],
                    name=f"{cat} (Fatalities)",
                    marker=dict(
                        color=colors_map.get(cat, '#333333'),
                        line=dict(color=colors_map.get(cat, '#333333'), width=1)
                    ),
                    opacity=0.4,
                    hovertemplate='Fatalities: %{y:,.0f}<extra></extra>',
                    legendgroup=cat,
                    showlegend=False
                ), secondary_y=True)

        fig_trend.update_layout(
            barmode='stack',
            title=dict(
                text="<b>Year-by-Year Trend of Disobedience Events and Fatalities in Iran</b><br>" +
                     "<span style='font-size:13px; color:gray;'>Solid Lines: Number of Events (Left Axis) | Transparent Stacked Bars: Number of Fatalities (Right Axis)</span>",
                font=dict(size=18, family="sans-serif"),
                x=0.5,
                xanchor='center'
            ),
            plot_bgcolor='white',
            paper_bgcolor='#fafafa',
            hovermode='x unified',
            legend=dict(
                title=dict(text="<b>Event Type [click to hide] </b>", font=dict(size=13)),
                orientation="h",
                yanchor="bottom",
                y=-0.4,
                xanchor="center",
                x=0.5,
                bgcolor="rgba(255, 255, 255, 0.8)",
                bordercolor="lightgray",
                borderwidth=1
            ),
            margin=dict(t=100, b=100, l=60, r=60)
        )

        fig_trend.update_xaxes(
            title_text='<b>Year</b>',
            showline=True, linewidth=1.5, linecolor='darkgray', 
            showgrid=False, tickmode='linear', dtick=1
        )
        
        fig_trend.update_yaxes(
            title_text="<b>Number of Events</b>",
            title_font=dict(color='#333333', size=14),
            showline=True, linewidth=1.5, linecolor='darkgray', 
            showgrid=True, gridcolor='#eaeaea', zeroline=False,
            tickfont=dict(color='#333333'),
            secondary_y=False
        )
        
        fig_trend.update_yaxes(
            title_text="<b>Number of Fatalities</b>",
            title_font=dict(color='gray', size=14),
            showline=False, 
            showgrid=False, zeroline=False,
            tickfont=dict(color='gray'),
            secondary_y=True
        )

        fig_trend.show()
        
        # Output to HTML
        report_dir = os.path.join(project_path, "Report")
        os.makedirs(report_dir, exist_ok=True)
        trend_html_path = os.path.join(report_dir, "Disobedience_Trend_Chart.html")
        fig_trend.write_html(trend_html_path)
        print(f"Trend chart successfully exported to: {trend_html_path}")
        
        # Store the HTML component of the sub-event trend chart so it can be injected into the Dashboard later
        sub_event_trend_chart_html = fig_trend.to_html(full_html=False, include_plotlyjs=False)
    else:
        print("Date column 'WEEK' not found for year extraction.")
else:
    print(f"Data file not found: {out_disobedience_path}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import os

# Using the path you provided
# Assuming project_path is defined earlier in your script
out_disobedience_path = os.path.join(project_path, "Data", "Data_Processed", "Iran_disobedience.csv")

if os.path.exists(out_disobedience_path):
    df = pd.read_csv(out_disobedience_path)
    df['Year'] = pd.to_datetime(df['WEEK']).dt.year

    # Aggregate specifically for the 'Protest' category to show the core friction
    protest_data = df[df['EVENT_TYPE'] == 'Protests'].groupby('Year').agg({
        'EVENTS': 'sum', 
        'FATALITIES': 'sum'
    }).reset_index()

    # Create a Professional Dual-Axis Dashboard
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # 1. Event Frequency (The "Social Pressure") - Subtle Area Chart
    fig.add_trace(go.Scatter(
        x=protest_data['Year'], 
        y=protest_data['EVENTS'],
        name="Protest Frequency",
        mode='lines',
        fill='tozeroy',
        line=dict(color='#3498db', width=1),
        fillcolor='rgba(52, 152, 219, 0.15)',
        stackgroup='one' 
    ), secondary_y=False)

    # 2. Fatalities (The "Lethality Surge") - High-Impact Bar Chart
    fig.add_trace(go.Bar(
        x=protest_data['Year'], 
        y=protest_data['FATALITIES'],
        name="Lethal Outcomes (Deaths)",
        marker=dict(
            color='#e74c3c',
            line=dict(color='#c0392b', width=1)
        ),
        opacity=0.85
    ), secondary_y=True)

    # 3. Add the 2024 "Pivot Point" Annotation
    fig.add_vline(x=2024, line_width=2, line_dash="dash", line_color="#2c3e50")
    
    # Textual Annotation for the Narrative
    fig.add_annotation(
        x=2024.2, y=protest_data['EVENTS'].max(),
        text="<b>2024 STRATEGIC PIVOT</b><br>State shifts to high-lethality<br>suppression tactics.",
        showarrow=False,
        align="left",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#2c3e50",
        borderwidth=1,
        font=dict(size=11)
    )

    # Professional Styling
    fig.update_layout(
        title=dict(
            text="<b>The Lethality Gap: Iran's Domestic Conflict (2016-2026)</b><br><span style='font-size:12px; color:gray;'>Contrasting the volume of civil protest against the escalation of lethal force</span>",
            x=0.05,
            font=dict(size=20, family="Arial")
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        hovermode='x unified',
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1),
        margin=dict(t=130, b=80, l=60, r=60)
    )

    # Axis Formatting
    fig.update_xaxes(title="Year", showgrid=False, dtick=1, linecolor='lightgray')
    fig.update_yaxes(title="<b>Total Protest Events</b>", secondary_y=False, showgrid=True, gridcolor='#f0f0f0')
    fig.update_yaxes(title="<b>Reported Fatalities</b>", secondary_y=True, showgrid=False)

    fig.show()
    
    # Save to your report directory
    report_path = os.path.join(project_path, "Report", "Lethality_Gap_Analysis.html")
    fig.write_html(report_path)
    print(f"Professional visualization exported to: {report_path}")

else:
    print("File not found at specified path.")

# 2. Iran Disobedience Level by Region

In [ ]:
import os
import pandas as pd

# Define paths (paths were previously defined, but just in case we redefine or use existing)
data_raw_path = os.path.join(project_path, "Data", "Data_Raw")
data_processed_path = os.path.join(project_path, "Data", "Data_Processed")

disobedience_file_name = "Middle-East_aggregated_data_up_to_week_of-2026-04-04.xlsx"
disobedience_file_path = os.path.join(data_raw_path, disobedience_file_name)
out_disobedience_path = os.path.join(data_processed_path, "Iran_disobedience.csv")

if os.path.exists(disobedience_file_path):
    print("Loading datasets...")
    # Load the big dataset
    df_disobedience = pd.read_excel(disobedience_file_path)
    
    # 1. Filter the dataset where the Country column is strictly equal to 'Iran'
    # Find the exact name of the country column (could be 'COUNTRY', 'Country', 'country')
    country_col = [c for c in df_disobedience.columns if c.lower() == 'country']
    
    if country_col:
        country_col_name = country_col[0]
        # Filter rows
        df_iran_disob = df_disobedience[df_disobedience[country_col_name] == 'Iran'].copy()
        
        # 2. Export to Data_Processed/Iran_disobedience.csv
        df_iran_disob.to_csv(out_disobedience_path, index=False)
        print(f"File successfully saved to: {out_disobedience_path}")
        print(f"Total rows extracted for Iran: {len(df_iran_disob)}")
        
        # 3. List unique items under the specified columns
        columns_to_list = ['ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'DISORDER_TYPE']
        
        for col in columns_to_list:
            # Match the exact column name case in the DataFrame
            actual_col = [c for c in df_iran_disob.columns if c.lower() == col.lower()]
            if actual_col:
                col_name = actual_col[0]
                unique_items = df_iran_disob[col_name].dropna().unique().tolist()
                print(f"\n--- Unique items in '{col_name}' ---")
                for item in sorted(unique_items):
                    print(f" - {item}")
            else:
                print(f"\n[Warning] Column '{col}' not found in the dataset.")
                
    else:
        print("Country column not found in the dataset.")
else:
    print(f"Disobedience data file not found at: {disobedience_file_path}")

In [ ]:
import os
import pandas as pd
import numpy as np
import json

# 1. Read the processed disobedience file
out_disobedience_path = os.path.join(project_path, "Data", "Data_Processed", "Iran_disobedience.csv")

if os.path.exists(out_disobedience_path):
    df_disob = pd.read_csv(out_disobedience_path)
    
    # 2. Define Weights
    weights_map = {
        'Peaceful protest': 1,
        'Protest with intervention': 2,
        'Excessive force against protesters': 3,
        'Violent demonstration': 4
    }
    
    weight_keys = [k.lower() for k in weights_map.keys()]
    
    # Filter only rows with the specified sub_event_types
    df_filtered = df_disob[df_disob['SUB_EVENT_TYPE'].str.lower().isin(weight_keys)].copy()
    
    if not df_filtered.empty:
        # Create Weight column
        def get_weight(sub_event):
            for k, v in weights_map.items():
                if k.lower() == str(sub_event).lower():
                    return v
            return 0
            
        df_filtered['weight'] = df_filtered['SUB_EVENT_TYPE'].apply(get_weight)
        
        # Calculate Disobedience
        pop_exp = np.where(df_filtered['POPULATION_EXPOSURE'] == 0, 1, df_filtered['POPULATION_EXPOSURE'])
        # 方案一：伤亡人数为0时，系数为1；有伤亡时，系数变为 1 + FATALITIES
        df_filtered['DISOBEDIENCE'] = (df_filtered['weight']**2 * df_filtered['EVENTS'] * (df_filtered['FATALITIES'] + 1)) / pop_exp
        
        
        # 3. Aggregate Data (Regional Aggregation)
        agg_funcs = {
            'DISOBEDIENCE': 'sum',
            'EVENTS': 'sum',
            'CENTROID_LATITUDE': 'mean',
            'CENTROID_LONGITUDE': 'mean'
        }
        df_agg = df_filtered.groupby('ADMIN1').agg(agg_funcs).reset_index()
        df_agg.rename(columns={
            'DISOBEDIENCE': 'TOTAL_SCORE',
            'EVENTS': 'TOTAL_EVENTS',
            'CENTROID_LATITUDE': 'LAT', 
            'CENTROID_LONGITUDE': 'LONG'
        }, inplace=True)
        
        # Normalize SCORE to 0-1 scale to act as an Index
        max_score = df_agg['TOTAL_SCORE'].max()
        #方案一：df_agg['DISOBEDIENCE_INDEX'] = df_agg['TOTAL_SCORE'] / max_score if max_score > 0 else 0
        #方案二：使用 np.log1p (即 ln(1+x)) 来进行对数平滑，然后再归一化
        log_scores = np.log1p(df_agg['TOTAL_SCORE'])
        max_log_score = log_scores.max()
        df_agg['DISOBEDIENCE_INDEX'] = log_scores / max_log_score if max_log_score > 0 else 0

        # 4. Calculate Percentage Distribution of Event Types
        type_counts = df_filtered.groupby(['ADMIN1', 'SUB_EVENT_TYPE']).size().unstack(fill_value=0)
        
        rename_map = {
            'Peaceful protest': 'peaceful',
            'Protest with intervention': 'intervention',
            'Excessive force against protesters': 'force',
            'Violent demonstration': 'violent'
        }
        
        for original_name, new_name in rename_map.items():
            if original_name not in type_counts.columns:
                type_counts[original_name] = 0
                
        type_counts = type_counts[list(rename_map.keys())]
        type_counts.rename(columns=rename_map, inplace=True)
        
        type_counts['total_events_pie'] = type_counts.sum(axis=1)
        for col in rename_map.values():
            type_counts[col] = (type_counts[col] / type_counts['total_events_pie'].replace(0, 1)) * 100
            type_counts[col] = type_counts[col].round(2)
            
        type_counts.drop(columns=['total_events_pie'], inplace=True)
        type_counts.reset_index(inplace=True)
        
        # 5. Merge Data for JS Plotting
        plot_data = pd.merge(df_agg, type_counts, on='ADMIN1')
        plot_data['TOTAL_SCORE'] = plot_data['TOTAL_SCORE'].round(4)
        plot_data['DISOBEDIENCE_INDEX'] = plot_data['DISOBEDIENCE_INDEX'].round(4)
        react_data_json = plot_data.to_json(orient='records')
        
        # 6. Generate Vanilla HTML with Leaflet and CSS Pie Charts
        html_template = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Interactive Regional Disobedience Index across Iran</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <style>
        body { margin: 0; padding: 0; font-family: Arial, sans-serif; background-color: white; color: #2a3f5f; }
        #map { width: 100vw; height: 100vh; position: absolute; top: 0; left: 0; z-index: 1; }
        
        .plotly-title {
            position: absolute; top: 20px; left: 0; width: 100%; text-align: center;
            z-index: 1000; pointer-events: none;
        }
        .plotly-title h1 { margin: 0; font-size: 24px; font-weight: normal; color: #2a3f5f; }

        .legend-panel {
            position: absolute; top: 50%; right: 30px; transform: translateY(-50%);
            z-index: 1000; padding: 15px; border-radius: 4px;
            display: flex; flex-direction: column; align-items: center; font-size: 12px; font-family: Arial, sans-serif;
            background: rgba(255,255,255,0.85); box-shadow: 0 2px 8px rgba(0,0,0,0.1); pointer-events: none;
        }

        .colorbar-title { font-weight: bold; margin-bottom: 8px; text-align: center; color: #444; }
        .colorbar-scale { display: flex; height: 150px; }
        .colorbar-labels {
            display: flex; flex-direction: column; justify-content: space-between; 
            height: 100%; margin-right: 8px; text-align: right; font-size: 11px;
        }
        .colorbar-gradient {
            width: 20px; height: 100%;
            background: linear-gradient(to top, #fff5f0, #fee0d2, #fcbba1, #fc9272, #fb6a4a, #ef3b2c, #cb181d, #99000d);
            border: 1px solid #ccc;
        }

        .info-panel {
            position: absolute; top: 20px; left: 60px; z-index: 1000;
            background: rgba(255,255,255,0.9); padding: 12px 20px; border-radius: 4px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1); border: 1px solid #e1e8ed; pointer-events: none;
        }
        .info-panel p { margin: 5px 0; font-size: 12px; color: #555; }

        .leaflet-tooltip.custom-tooltip {
            background: rgba(255, 255, 255, 0.98); border: 1px solid #ddd;
            box-shadow: 0 4px 15px rgba(0,0,0,0.15); border-radius: 4px; padding: 15px; font-family: Arial, sans-serif;
        }
        .tooltip-content { text-align: center; width: 170px; }
        .tooltip-content h4 { margin: 0 0 5px 0; font-size: 15px; color: #2a3f5f; }
        .tooltip-content .region-label { font-size: 11px; color: #888; text-transform: uppercase; letter-spacing: 1px; margin: 0 0 8px 0; }
        .tooltip-content .score-label { margin: 0 0 10px 0; font-size: 13px; color: #444; font-weight: bold; }
        
        .pie-chart { width: 100px; height: 100px; border-radius: 50%; margin: 10px auto; box-shadow: 0 2px 5px rgba(0,0,0,0.2); }
        .legend { display: flex; flex-wrap: wrap; width: 100%; font-size: 11px; margin-top: 15px; text-align: left; }
        .legend-item { display: flex; align-items: center; width: 50%; margin-bottom: 6px; color: #444; }
        .color-box { width: 12px; height: 12px; margin-right: 6px; border-radius: 2px; border: 1px solid #ccc;}
    </style>
</head>
<body>
    <div class="plotly-title"><h1>Interactive Regional Disobedience Index across Iran</h1></div>

    <div class="info-panel">
        <p><b>Bubble Size:</b> Total Events</p>
        <p><b>Bubble Color:</b> Disobedience Index (0 - 1.0)</p>
        <p><b>Hover:</b> Sub-event proportion Pie Chart</p>
    </div>

    <!-- Combined Legend Panel -->
    <div class="legend-panel">
        <div class="colorbar-title">Index<br>(Color)</div>
        <div class="colorbar-scale">
            <div class="colorbar-labels" id="colorbar-labels">
                <span>1.0</span>
                <span>0.75</span>
                <span>0.50</span>
                <span>0.25</span>
                <span>0</span>
            </div>
            <div class="colorbar-gradient"></div>
        </div>
    </div>

    <div id="map"></div>

    <script>
        var map = L.map('map', {zoomControl: false}).setView([32.4279, 53.6880], 5);
        L.control.zoom({ position: 'topleft' }).addTo(map);
        
        L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png', {
            attribution: '&copy; CARTO', subdomains: 'abcd', maxZoom: 20
        }).addTo(map);

        var mapData = __JSON_DATA__;
        
        // Sort data descending by TOTAL_EVENTS so large bubbles are rendered first (in the background)
        mapData.sort(function(a, b) {
            return b.TOTAL_EVENTS - a.TOTAL_EVENTS;
        });
        
        // Find MAX values for dynamic assignment
        var maxEvents = Math.max(...mapData.map(d => d.TOTAL_EVENTS));

        function getRadius(val, max) { return 5 + Math.sqrt(val / max) * 30; }

        // Compile Color mapping based entirely on Normalized Disobedience Index (0.0 to 1.0)
        function getRedColor(ratio) {
            var pal = ['#fff5f0', '#fee0d2', '#fcbba1', '#fc9272', '#fb6a4a', '#ef3b2c', '#cb181d', '#99000d'];
            var idx = Math.floor(ratio * (pal.length - 1));
            if(idx < 0) idx = 0; if(idx >= pal.length) idx = pal.length - 1;
            return pal[idx];
        }
        
        mapData.forEach(function(region) {
            var size = getRadius(region.TOTAL_EVENTS, maxEvents);
            var color = getRedColor(region.DISOBEDIENCE_INDEX);
            
            var circle = L.circleMarker([region.LAT, region.LONG], {
                radius: size, fillColor: color, color: "#cb181d",   
                weight: 1.5, opacity: 0.8, fillOpacity: 0.75
            }).addTo(map);

            circle.on('mouseover', function() {
                this.setStyle({ fillOpacity: 1, weight: 2.5, color: "#333" });
                this.bringToFront();
            });
            circle.on('mouseout', function() {
                this.setStyle({ fillOpacity: 0.75, weight: 1.5, color: "#cb181d" });
                // Restore sorting order specifically so large bubbles don't get stuck on top after hovering
                mapData.forEach(function(r) { /* Can be handled passively, or simple z-index layering if needed*/ });
            });

            var p = region.peaceful || 0, i = region.intervention || 0, f = region.force || 0, v = region.violent || 0;
            var c_p = '#3498db', c_i = '#f39c12', c_f = '#8e44ad', c_v = '#e74c3c';

            var bg = 'conic-gradient(' + 
                c_p + ' 0% ' + p + '%, ' +                             
                c_i + ' ' + p + '% ' + (p+i) + '%, ' +                 
                c_f + ' ' + (p+i) + '% ' + (p+i+f) + '%, ' +           
                c_v + ' ' + (p+i+f) + '% 100%)';                       

            var tooltipHtml = `<div class="tooltip-content">
                <p class="region-label">Region / Province</p>
                <h4>${region.ADMIN1}</h4>
                <p class="score-label">Events: ${region.TOTAL_EVENTS}<br>Index: ${region.DISOBEDIENCE_INDEX.toFixed(2)}</p>
                <div class="pie-chart" style="background: ${bg};"></div>
                <div class="legend">
                    <div class="legend-item"><div class="color-box" style="background:${c_p};"></div>Peaceful</div>
                    <div class="legend-item"><div class="color-box" style="background:${c_i};"></div>Intervention</div>
                    <div class="legend-item"><div class="color-box" style="background:${c_f};"></div>Force</div>
                    <div class="legend-item"><div class="color-box" style="background:${c_v};"></div>Violent</div>
                </div>
            </div>`;

            circle.bindTooltip(tooltipHtml, { direction: 'top', className: 'custom-tooltip', offset: [0, -size] });
        });
    </script>
</body>
</html>"""

        html_content = html_template.replace("__JSON_DATA__", react_data_json)
        
        report_dir = os.path.join(project_path, "Report")
        os.makedirs(report_dir, exist_ok=True)
        html_export_path = os.path.join(report_dir, "Iran_Disobedience_Report.html")
        
        with open(html_export_path, "w", encoding="utf-8") as f:
            f.write(html_content)
        
        print(f"Data and Map successfully exported!")
        print(f"-> HTML Generated: {html_export_path}")
        print(f"-> Size maps to 'Events', Color maps to Normalized 'Index(0-1)'")
        
    else:
        print("No matching sub-event types found in the dataset.")
else:
    print(f"File not found at: {out_disobedience_path}")

# 3. Create Webpage

In [ ]:
# Dashboard HTML Template
html_dashboard_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Iranian Civil Disobedience Dashboard</title>
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        :root {
            --bg-color: #f7f9fa;
            --text-primary: #14171a;
            --text-secondary: #657786;
            --card-bg: white;
            --border-color: #e1e8ed;
        }
        
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            margin: 0;
            padding: 0;
            background-color: var(--bg-color);
            color: var(--text-primary);
            line-height: 1.6;
        }

        .header {
            background-image: linear-gradient(rgba(0, 0, 0, 0.6), rgba(0, 0, 0, 0.8)), url('https://images.wsj.net/im-88861666?width=1280&height=853');
            background-size: cover;
            background-position: center;
            padding: 4rem 0;
            border-bottom: 1px solid var(--border-color);
            text-align: center;
            color: white;
        }

        .header h1 {
            margin: 0;
            font-size: 2.2rem;
            color: white;
            text-shadow: 1px 1px 3px rgba(0,0,0,0.8);
        }

        .header p {
            color: #e1e8ed;
            max-width: 800px;
            margin: 1rem auto 0;
            text-shadow: 1px 1px 3px rgba(0,0,0,0.8);
        }

        .container {
            max-width: 1200px;
            margin: 0 auto;
            padding: 2rem;
        }

        .narrative {
            background: var(--card-bg);
            padding: 2rem;
            border-radius: 8px;
            margin-bottom: 2rem;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }

        .narrative h2 {
            margin-top: 0;
            color: #2b3a4a;
        }
        
        .map-wrapper {
            background: var(--card-bg);
            padding: 1rem;
            border-radius: 8px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }

        #map {
            height: 600px;
            width: 100%;
            border-radius: 6px;
        }

        .footnotes {
            margin-top: 3rem;
            padding-top: 1rem;
            border-top: 1px solid var(--border-color);
            font-size: 0.9rem;
            color: var(--text-secondary);
        }

        /* Tooltip Styles */
        .tooltip-content {
            font-family: inherit;
            padding: 5px;
            min-width: 200px;
        }
        .tooltip-content h4 {
            margin: 0 0 10px 0;
            font-size: 16px;
            color: #333;
            border-bottom: 1px solid #ddd;
            padding-bottom: 5px;
        }
        .region-label {
            font-size: 10px;
            text-transform: uppercase;
            color: #777;
            margin: 0 0 2px 0;
        }
        .score-label {
            font-weight: bold;
            font-size: 13px;
            margin-bottom: 15px;
        }
        .pie-chart {
            width: 100%;
            height: 120px;
            border-radius: 50%;
            margin-bottom: 15px;
            box-shadow: 0 0 5px rgba(0,0,0,0.2) inset;
        }
        .legend {
            display: grid; 
            grid-template-columns: 1fr 1fr; /* 分成两列，解决挤压问题 */
            gap: 8px; 
            width: 100%; 
            font-size: 11px; 
            margin-top: 15px; 
            text-align: left;
        }
        .legend-item {
            display: flex; 
            align-items: center; 
            color: #444; 
            white-space: nowrap; /* 强制不换行，防止文字重叠 */
        }
        .color-box {
            width: 10px;
            height: 10px;
            margin-right: 6px;
            border-radius: 2px;
            flex-shrink: 0; /* 防止小方块被挤扁 */
        }
    </style>
</head>
<body>

<div class="header">
    <h1>Betting on the Street: Can Iranian People Dissent Flip the House Odds?</h1>
    <p>A geospatial analysis of political violence, demonstrations, and shifting modes of unrest from 2016 to 2026. This dashboard examines regional hotspots and the transition in event types.</p>
</div>

<div class="container">
    <div class="narrative">
        <h2 style="border-left: 4px solid #cb181d; padding-left: 15px; color: #2a3f5f;">Introduction</h2>
        <p>The ongoing US-Iranian conflict has disturbed the global economy, driving fuel prices to rise substantially. Despite various diplomatic efforts, a lasting resolution remains elusive. 
        Following the assassination of Ali Khamenei on 28 February, the expectation of Iranian regime change has reached a fever pitch. It appears that the United States and Israel have deployed 
        strategic air strikes, political pressure, and the incitement of internal uprisings to destabilize or overthrow the IRGC regime. Unfortunately, while the U.S. has destroyed Iran’s conventional 
        air and naval capabilities, the IRGC regime maintains its grip through the control of the Strait of Hormuz and brutal suppression of domestic unrest. </p>
        <p>Liberal hawks such as Kenneth M. Pollack advocate for moves such as direct air support to an Iranian popular revolt to increase the odds of winning the conflict. According to him, Trump's 
        military campaign gambles on 'whether a massive air campaign can trigger a popular rebellion that takes down the regime in Tehran.'<sup id="ref1"><a href="#fn1">[1]</a></sup></p>
        <p>It thus becomes vital to study the Iranian people's disobedience tendency and their resistance to the government. The author examines the patterns in civil disobedience and resistance 
        movements across Iran in the last decade. This report also aims to use geospatial data science to identify strategic clues: which regions are currently experiencing the highest friction, which 
        cities serve as the most viable starting points for unrest, and where the regime's control is most contested.</p>
        
        <h2 style="border-left: 4px solid #cb181d; padding-left: 15px; color: #2a3f5f;">An Underlying Tide of Resistance</h2>
        <p>The interactive trend charts below trace this explosion of domestic political violence and fatalities, marking 2024 as a critical turning point. That is when the Iran–Israel proxy conflict 
        escalated to a series of direct confrontations. The Israeli strikes were captured in the sharp increase in the number of poltical violence events and number of total fatalities. <sup id="ref2"><a href="#fn2">[2]</a></sup></p>
        <p>Domestically, the intesified conflict and economic failure has led to a significant increase in citizen dissent. Strickingly, the data reveals a significant structural shift in the nature 
        of domestic unrest starting in 2024. <strong>While outward signs of demonstration have declined, indicators of organized resistance have rised significantly.</strong> The sharp decline in demonstration 
        events after 2024 does not suggest a restoration of social stability or a decrease in public grievances. Instead, it reflects a "chilling effect" resulting from extreme state suppression. The high cost 
        of public assembly—characterized by mass arrests and lethal force—has forced the peaceful demonstrators to retreat from the streets.<p>
        <p>Critically, the upward trend in strategic developments provides strong circumstantial evidence that opposition forces are undergoing a qualitative upgrade. In ACLED terms, "Strategic Developments"
        often involve recruitment, weapon movements, and the establishment of underground networks. The rise in this metric suggests that the opposition is moving away from uncoordinated street rallies toward 
        organized, clandestine resistance.</p>
        
        <!-- Inject Plotly interactive chart here -->
        <div class="trend-chart-container" style="margin: 2rem 0; width: 100%; border: 1px solid #e1e8ed; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); background: white;">
            __TREND_CHART_HTML__
        </div>
    <div>

        <p>Consequently, the shift from visible public squares to clandestine networks marks the <strong>beginning of a more entrenched and dangerous phase of the conflict</strong>. The total number of demonstration events has dropped from 3,668 in 2024 to 2,131 by March 2026. 
        On the contrary, the number of violent demonstrations has increased from 4 in 2024 to 194 by March 2026, with fatalities increased from minimal levels to 22,669 by March 2026.
        As the regime closes all avenues for peaceful expression, the opposition's survival necessitates a transition toward asymmetric tactics and organizational resilience. 
        The visualization below deconstructs these trends into specific sub-categories of protests and demonstrations, highlighting the shrinking space for peaceful demonstrations and the corresponding rise in high-stakes confrontations.</p>
        
        <div class="trend-chart-container" style="margin: 2rem 0; width: 100%; border: 1px solid #FFFFFF; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); background: white;">
            __SUB_EVENT_TREND_CHART_HTML__
        </div>
    </div>
  
        <h2 style="border-left: 4px solid #cb181d; padding-left: 15px; color: #2a3f5f;">Regional Variance in Disobedience Levels</h2>
        <p>
        To move beyond simple event counts, this analysis develops a <strong>Disobedience Severity Index (DSI)</strong> to quantify the intensity of regional unrest. 
        By weighting events based on the degree of state-citizen friction, this index helps identify "breakthrough points" where the regime rule has significantly challenged. 
        The weights are assigned as follows:
        </p>
        <p><strong>1. Categorizing Resistance Sub-types</strong></p>
        
    

    <table class="dsi-table">
        <thead>
            <tr>
                <th>ACLED Event Type</th>
                <th>Description</th>
                <th>Weight (w)</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td><strong>Peaceful Protest</strong></td>
                <td>Non-violent assembly without state interference; represents baseline dissent.</td>
                <td>1</td>
            </tr>
            <tr>
                <td><strong>Protest with Intervention</strong></td>
                <td>Non-violent assembly met with state attempts to disperse or arrest participants.</td>
                <td>2</td>
            </tr>
            <tr>
                <td><strong>Excessive Force against Protesters</strong></td>
                <td>State actors use lethal or disproportionate force against non-violent protesters.</td>
                <td>3</td>
            </tr>
            <tr>
                <td><strong>Violent Demonstration</strong></td>
                <td>Kinetic events involving rioting or armed clashes; indicates a transition to active rebellion.</td>
                <td>4</td>
            </tr>
        </tbody>
    </table>

    <style>
        /* Styling for the Methodology Table */
        .dsi-table {
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
            background-color: white;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 2px 5px rgba(0,0,0,0.05);
        }
        
        .dsi-table th {
            background-color: #f1f3f5;
            color: #2c3e50;
            text-align: left;
            padding: 12px 15px;
            border-bottom: 2px solid #dee2e6;
            font-size: 0.9rem;
            text-transform: uppercase;
            letter-spacing: 0.05em;
        }

        .dsi-table td {
            padding: 12px 15px;
            border-bottom: 1px solid #edf2f7;
            font-size: 0.95rem;
            color: #4a5568;
            vertical-align: middle;
        }

        .dsi-table tr:last-child td {
            border-bottom: none;
        }

        /* Highlight the Weight Column */
        .dsi-table td:last-child {
            font-weight: bold;
            color: #e74c3c;
            text-align: center;
            background-color: #fffaf0;
        }

        .dsi-table tr:hover {
            background-color: #f8fafc;
        }
    </style>



    <h3 style="font-size: 1.1rem; color: #444;">2. Index Calculation</h3>
    <p>For each record, the <em>Disobedience Score</em> is calculated using the following formula. 
    A baseline of 1 is added to the fatalities to ensure non-lethal events retain their fundamental frequency and weight, 
    while still heavily amplifying the score for events with casualties, all normalized for population density:</p>
    
    <div style="background: #f4f4f4; padding: 20px; border-radius: 5px; text-align: center; font-family: 'Courier New', Courier, monospace; font-size: 1.2rem; margin: 20px 0;">
        Score = (w<sup>2</sup> &times; Events &times; (Fatalities + 1)) / Population Exposure
    </div>

    <p>The resulting scores are then aggregated by province (Admin1). To prevent extreme outliers—such as regions with 
    exceptionally high fatality rates—from visually suppressing the variance across other provinces, a logarithmic 
    transformation <em>[ln(1 + x)]</em> is applied to the aggregated provincial scores. Finally, these log-smoothed 
    scores are normalized to a 0.0 &ndash; 1.0 scale (where 1.0 represents the highest intensity recorded in 
    the dataset), creating a balanced and comparative Disobedience Index across Iran.</p>
    
    <h3 style="font-size: 1.1rem; color: #444; margin-top: 2rem;">3. Visual Interpretation & Geospatial Distribution</h3>
    <p style="color: #555;">
        The interactive map below translates the DSI methodology into a spatial intelligence tool. By analyzing the relationship between 
        volume and intensity, we can pinpoint the "breakthrough points" should any future military campaign aim to leverage domestic dissent.
    </p>
    

    <p>The map reveals a complex landscape of unrest across Iran. <strong>While major cities like Tehran and Isfahan
    record the highest absolute number of protests and demonstration events, Sistan and Baluchestan province
    stands out with the highest index severity</strong>.
    In central provinces, protests frequently erupt over economic grievances such as inflation, currency devaluation, 
    unpaid wages, and severe water shortages (e.g., the "Thirst Uprising" in Isfahan and Khuzestan). Given that these 
    regions are economic, political, and cultural centers where stability is highly prioritized, the regime's repression 
    often manifests as tear gas, water cannons, beatings, and mass arrests. Tehran, in particular, exhibits high proportion 
    of medium-level confrontation categorized by high proprotion of protests with intervention events</p>
    
    <p>In stark contrast, the brutal forceful repression in Sistan and Baluchestan province, home to the Baluch (predominantly Sunni) minority, 
    has long faced severe economic marginalization and systemic discrimination. The regime constantly employs highly militarized and lethal force 
    to suppress protests in the region, citing "separatism" and border security pressures. The exceptionally high fatality rates, compounded by the severe nature of the clashes, 
    indicate that the friction between the state and the local population has reached a critical threshold, making this region the most volatile in the country.</p>


   <div class="map-wrapper" style="position: relative; margin: 2rem 0; padding: 10px; background: white; border: 1px solid #e1e8ed; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
        
        <div class="map-toolbar" style="display: flex; justify-content: space-around; background: #f8fafc; padding: 10px; border-radius: 8px; margin-bottom: 10px; border: 1px solid #edf2f7; font-size: 0.85rem;">
            <span><b style="color:#657786;">●</b> <b>Size:</b> Total Events</span>
            <span><b style="color:#cb181d;">●</b> <b>Color:</b> Disobedience Index</span>
            <span><b style="color:#3498db;">●</b> <b>Hover:</b> Detail Pie Chart</span>
        </div>

        <div id="map" style="height: 600px; width: 100%; border-radius: 8px; z-index: 1;"></div>

        <div class="legend-panel" style="position: absolute; bottom: 50px; right: 25px; z-index: 1000; background: rgba(255, 255, 255, 0.9); padding: 15px; border-radius: 6px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); border: 1px solid #ddd; display: flex; flex-direction: column; align-items: center; pointer-events: none;">
            <div style="font-weight: bold; font-size: 11px; margin-bottom: 8px; text-align: center; color: #333;">Index<br>Severity</div>
            <div style="display: flex; height: 160px;">
                <div style="display: flex; flex-direction: column; justify-content: space-between; margin-right: 10px; text-align: right; font-size: 10px; color: #666; font-family: sans-serif;">
                    <span>1.0</span>
                    <span>0.75</span>
                    <span>0.50</span>
                    <span>0.25</span>
                    <span>0</span>
                </div>
                <div style="width: 15px; height: 100%; background: linear-gradient(to top, #fff5f0, #fee0d2, #fcbba1, #fc9272, #fb6a4a, #ef3b2c, #cb181d, #99000d); border: 1px solid #ccc; border-radius: 2px;"></div>
            </div>
        </div>
        
        <p style="margin-top: 1rem; font-size: 0.85rem; color: var(--text-secondary); text-align: center; font-style: italic;">
            Figure 1: Interactive Geospatial Analysis of the Disobedience Severity Index (DSI) across Iran (2016-2026).
        </p>
        
    </div>



    <div class="conclusion-section" style="margin-top: 3rem;">
        <h2 style="border-left: 4px solid #cb181d; padding-left: 15px; color: #2a3f5f;">Looking Ahead: Responsible Strategy Called</h2>
        <p>Combining the temporal trend analysis with the spatial mapping of the Disobedience Index reveals a nuanced framework for understanding the trajectory 
        of potential regime instability in Iran. Since 2024, the total number of peaceful demonstrations has declined significantly, replaced by more violent forms of resistance and 
        underground strategic development. The total fatalities of protesters skyrocketed in 2026, marking the increasing dissent among Iranian citizens and more brutal repression.</p>
        <p>The regional analysis highlights that Tehran and Isfahan are the most active centers of unrest, and Sistan and Baluchestan province has the highest intensity of disobedience
        and confrontation, driven by severe repression and long-standing grievances. These findings suggest that any external support for regime change should be strategically targeted.
        Instead of broad, indiscriminate support for a popular revolt, the focus should be on providing precision support to the most volatile regions, such as Sistan and Baluchestan, 
        while also bolstering the resilience of opposition networks in major cities. This approach would maximize the impact of support while minimizing the risks of provoking further 
        repression or alienating potential allies within the country.</p>
        <p>Ultimately, the path to regime change in Iran is fraught with complexity and uncertainty. By leveraging data-driven insights into the patterns of dissent and repression,
        policy-makers can ideally make more informed decisions about how to support the Iranian people's aspirations for freedom and justice. However, it is crucial to rethink what will 
        the path of the nation after regime change. Whether the separatist forces in Sistan and Baluchestan, gaining military support from the US, will be able to peacefully integrate into 
        the new regime or will further fragment the country is a question that requires careful consideration.</p>

    <div class="footnotes">
        <p id="fn1"><strong>[1]</strong> <em>How to Raise the Odds of Regime Change in Iran: America Can Make It Easier for Iranians to Revolt</em>, Kenneth M. Pollack, March 18, 2026, <a href="https://www.foreignaffairs.com/iran/how-raise-odds-regime-change-iran" target="_blank">Foreign Affairs</a>. <a href="#ref1">↩</a></p>
        <p id="fn2"><strong>[2]</strong> The data relies on millions of ACLED (Armed Conflict Location & Event Data Project) points up to March 2026. Sub-category filters for political violence and demonstration events were strictly aggregated to map regional severity. <a href="#ref2">↩</a></p>
        <p style="margin-top: 1rem;"><strong>Map Visualization Note:</strong> The region bubbles visualize peaceful protests (Blue), state interventions (Orange), excessive force (Purple), and violent clashes (Red). The exact percentage split is shown in the tooltip pie chart when hovering over regions.</p>
    </div>
</div>

<script>
    var mapData = __DASHBOARD_JSON_DATA__;
    
    // Initialize map focused on Iran
    var map = L.map('map').setView([32.4279, 53.6880], 5);
    
    L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png', {
        attribution: '&copy; CARTO', subdomains: 'abcd', maxZoom: 20
    }).addTo(map);
    
    mapData.sort(function(a, b) { return b.TOTAL_EVENTS - a.TOTAL_EVENTS; });
    
    var maxEvents = Math.max(...mapData.map(d => d.TOTAL_EVENTS));
    function getRadius(val, max) { return 5 + Math.sqrt(val / max) * 30; }

    function getRedColor(ratio) {
        var pal = ['#fff5f0', '#fee0d2', '#fcbba1', '#fc9272', '#fb6a4a', '#ef3b2c', '#cb181d', '#99000d'];
        var idx = Math.floor(ratio * (pal.length - 1));
        if(idx < 0) idx = 0; if(idx >= pal.length) idx = pal.length - 1;
        return pal[idx];
    }
    
    mapData.forEach(function(region) {
        var size = getRadius(region.TOTAL_EVENTS, maxEvents);
        var color = getRedColor(region.DISOBEDIENCE_INDEX);
        
        var circle = L.circleMarker([region.LAT, region.LONG], {
            radius: size, fillColor: color, color: "#cb181d",   
            weight: 1.5, opacity: 0.8, fillOpacity: 0.75
        }).addTo(map);

        circle.on('mouseover', function() {
            this.setStyle({ fillOpacity: 1, weight: 2.5, color: "#333" });
            this.bringToFront();
        });
        circle.on('mouseout', function() {
            this.setStyle({ fillOpacity: 0.75, weight: 1.5, color: "#cb181d" });
        });

        var p = region.peaceful || 0, i = region.intervention || 0, f = region.force || 0, v = region.violent || 0;
        var c_p = '#3498db', c_i = '#f39c12', c_f = '#8e44ad', c_v = '#e74c3c';

        var bg = 'conic-gradient(' + 
            c_p + ' 0% ' + p + '%, ' +                             
            c_i + ' ' + p + '% ' + (p+i) + '%, ' +                 
            c_f + ' ' + (p+i) + '% ' + (p+i+f) + '%, ' +           
            c_v + ' ' + (p+i+f) + '% 100%)';                       

        var tooltipHtml = `<div class="tooltip-content">
            <p class="region-label">Region / Province</p>
            <h4>${region.ADMIN1}</h4>
            <p class="score-label">Events: ${region.TOTAL_EVENTS}<br>Index: ${region.DISOBEDIENCE_INDEX.toFixed(2)}</p>
            <div class="pie-chart" style="background: ${bg};"></div>
            <div class="legend">
                <div class="legend-item"><div class="color-box" style="background:${c_p};"></div>Peaceful</div>
                <div class="legend-item"><div class="color-box" style="background:${c_i};"></div>Intervention</div>
                <div class="legend-item"><div class="color-box" style="background:${c_f};"></div>Force</div>
                <div class="legend-item"><div class="color-box" style="background:${c_v};"></div>Violent</div>
            </div>
        </div>`;

        circle.bindTooltip(tooltipHtml, {
            sticky: true,
            className: 'custom-tooltip',
            opacity: 0.95
        });
    });
</script>
</body>
</html>
"""

# Ensure react_data_json from the previous cell is used
dashboard_html_content = html_dashboard_template.replace("__DASHBOARD_JSON_DATA__", react_data_json).replace("__TREND_CHART_HTML__", trend_chart_html).replace("__SUB_EVENT_TREND_CHART_HTML__", sub_event_trend_chart_html)
dashboard_out_path = os.path.join(project_path, "Report", "Dashboard.html")

with open(dashboard_out_path, "w", encoding="utf-8") as f:
    f.write(dashboard_html_content)

print(f"Dashboard successfully created and exported to: {dashboard_out_path}")

Dashboard successfully created and exported to: /content/drive/MyDrive/Study/HKU/POLI3148 Data Science in Politics and Public Administration/Assignments/Assignment 1/Report/Dashboard.html
